In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Read the dataset
url = "https://raw.githubusercontent.com/basilatawneh/Students-Academic-Performance-Dataset-xAPI-Edu-Data-/master/xAPI-Edu-Data.csv"
df = pd.read_csv(url)

In [ ]:
# 2. Initial inspection

In [5]:
print("First 10 rows:")
print(df.head(10))

print("\nShape of dataset:")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nBasic information:")
print(df.info())

print("\nSummary statistics:")
print(df.describe(include='all'))

First 10 rows:
  gender NationalITy PlaceofBirth       StageID GradeID SectionID Topic  \
0      M          KW       KuwaIT    lowerlevel    G-04         A    IT   
1      M          KW       KuwaIT    lowerlevel    G-04         A    IT   
2      M          KW       KuwaIT    lowerlevel    G-04         A    IT   
3      M          KW       KuwaIT    lowerlevel    G-04         A    IT   
4      M          KW       KuwaIT    lowerlevel    G-04         A    IT   
5      F          KW       KuwaIT    lowerlevel    G-04         A    IT   
6      M          KW       KuwaIT  MiddleSchool    G-07         A  Math   
7      M          KW       KuwaIT  MiddleSchool    G-07         A  Math   
8      F          KW       KuwaIT  MiddleSchool    G-07         A  Math   
9      F          KW       KuwaIT  MiddleSchool    G-07         B    IT   

  Semester Relation  raisedhands  VisITedResources  AnnouncementsView  \
0        F   Father           15                16                  2   
1        F   

In [ ]:
# 3. Handle inconsistencies

In [6]:
# Strip leading/trailing spaces from text columns
for col in df.select_dtypes(include= 'object').columns:
  df[col] = df[col].astype(str).str.strip()

In [7]:
# Replace common placeholder missing values with NaN
df.replace(['?', 'NA', 'N/A', 'na', 'null', 'None', ''], np.nan, inplace = True)

In [ ]:
# 4. Check missing values

In [8]:
print("\nMissing values in each column:")
print(df.isnull().sum())


Missing values in each column:
gender                      0
NationalITy                 0
PlaceofBirth                0
StageID                     0
GradeID                     0
SectionID                   0
Topic                       0
Semester                    0
Relation                    0
raisedhands                 0
VisITedResources            0
AnnouncementsView           0
Discussion                  0
ParentAnsweringSurvey       0
ParentschoolSatisfaction    0
StudentAbsenceDays          0
Class                       0
dtype: int64


In [ ]:
# 5. Fill missing values

In [9]:
# Numeric columns -> median
numeric_cols = df.select_dtypes(include= [np.number]).columns
for col in numeric_cols:
  df[col].fillna(df[col].median(), inplace = True)

/tmp/ipykernel_918/3886543270.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace = True)


In [11]:
# Categorical columns -> mode
categorical_cols = df.select_dtypes(include = 'object').columns
for col in categorical_cols:
  df[col].fillna(df[col].mode()[0], inplace = True)

/tmp/ipykernel_918/186878642.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace = True)


In [12]:
print("\nMissing values after handling:")
print(df.isnull().sum())


Missing values after handling:
gender                      0
NationalITy                 0
PlaceofBirth                0
StageID                     0
GradeID                     0
SectionID                   0
Topic                       0
Semester                    0
Relation                    0
raisedhands                 0
VisITedResources            0
AnnouncementsView           0
Discussion                  0
ParentAnsweringSurvey       0
ParentschoolSatisfaction    0
StudentAbsenceDays          0
Class                       0
dtype: int64


In [13]:
# 6. Remove duplicate rows

print("\nDuplicated rows before removal:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Duplicate rows after removal:", df.duplicated().sum())


Duplicated rows before removal: 2
Duplicate rows after removal: 0


In [16]:
# 7. Scan numeric variables for outliers using IQR

print("\nOutlier detection using IQR:")
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1

  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  print(f"{col}: {len(outliers)} outliers")

  # Handle outliers by capping
  df[col] = df[col].clip(lower_bound, upper_bound)


Outlier detection using IQR:
raisedhands: 0 outliers
VisITedResources: 0 outliers
AnnouncementsView: 0 outliers
Discussion: 0 outliers


In [17]:
# 8. Apply data transformation

# Log transformation to reduce skewness in one variable
# Using log1p to safely handle zero values
if 'VisITedResources' in df.columns:
    df['log_VisITedResources'] = np.log1p(df['VisITedResources'])

In [18]:
print("\nData after transformation:")
print(df[['VisITedResources', 'log_VisITedResources']].head())


Data after transformation:
   VisITedResources  log_VisITedResources
0                16              2.833213
1                20              3.044522
2                 7              2.079442
3                25              3.258097
4                50              3.931826


In [19]:
# 9. Final summary
print("\nFinal dataset shape:")
print(df.shape)

print("\nFinal data types:")
print(df.dtypes)


Final dataset shape:
(478, 18)

Final data types:
gender                       object
NationalITy                  object
PlaceofBirth                 object
StageID                      object
GradeID                      object
SectionID                    object
Topic                        object
Semester                     object
Relation                     object
raisedhands                   int64
VisITedResources              int64
AnnouncementsView             int64
Discussion                    int64
ParentAnsweringSurvey        object
ParentschoolSatisfaction     object
StudentAbsenceDays           object
Class                        object
log_VisITedResources        float64
dtype: object
